In [1]:
from cloudpendulumclient.client import Client

In [2]:
user_token = "1747283276843825152"

In [3]:

import math
import time
import subprocess
from pathlib import Path

import numpy as np
import wget


def convert_flv_to_mp4(input_path, output_path):
    """Convert an FLV file to MP4 using ffmpeg (no re-encode)."""
    command = [
        "ffmpeg", "-i", input_path,
        "-c:v", "copy", "-c:a", "copy",
        output_path,
    ]
    process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if process.returncode == 0:
        print(f"Conversion successful: {output_path}")
    else:
        print(f"Error during conversion: {process.stderr.decode()}")


def run_free_fall(tf, dt, x0,
                  user_token=None,
                  experiment_type="DoublePendulum",
                  preparation_time=0.0,
                  record=False):
    """Run a passive (zero-torque) experiment from initial state x0.

    Parameters
    ----------
    tf : float
        Total recording time (s) after release.
    dt : float
        Control-loop step (s).
    x0 : array-like of length 2 or 4
        Initial state. If length 2, treated as [q1, q2]. If length 4,
        only the first two entries (positions) are sent to the cloud;
        the velocities at release are whatever the hardware produces.
    user_token : str or None
        Cloud token. If None, falls back to the local pyCandle client.
    experiment_type : str
    preparation_time : float
        Time (s) the hardware spends driving the pendulum to x0 before release.
    record : bool
        If True, downloads the experiment video and converts it to MP4.

    Returns
    -------
    t_values        : (N,)   measured timestamps
    x_values        : (N, 4) [q1, q2, q1_dot, q2_dot] columns
    tau_values      : (N, 2) measured motor torques (~0 except residuals)
    des_tau_values  : (N, 2) commanded torque (exactly zero everywhere)
    vod_filepath    : str or None
    cell_id         : str
    """
    if user_token is None:
        from cloud_pendulum_local import Client
        user_token = ""
    else:
        from cloudpendulumclient.client import Client

    # The helper that the original code relies on for camera detection.
    # If extract_cell_id lives in your simulation module, change this import accordingly.
    from double_pendulum.simulation.simulation import extract_cell_id

    c = Client()

    # Only positions are sent as the initial state.
    # x0 = np.asarray(x0, dtype=float)
    initial_state = x0.copy()
    initial_state = initial_state[0:len(initial_state)//2]

    session_token, live_url = c.start_experiment(
        user_token, experiment_type, tf,
        preparation_time=preparation_time, record=record,
        initial_position=initial_state,
    )
    print("Your session token is:", session_token)
    cell_id = extract_cell_id(live_url)
    print("Detected cell/camera:", cell_id)

    vod_filepath = None

    # Pure passive: zero impedance, zero commanded torque from step zero.
    c.set_impedance_controller_params(0.0, 0.0, session_token)

    n = int(tf / dt)
    print(f"n:{n}")

    meas_time_vec     = np.zeros(n)
    meas_pos_shoulder = np.zeros(n)
    meas_vel_shoulder = np.zeros(n)
    meas_tau_shoulder = np.zeros(n)
    des_tau_shoulder  = np.zeros(n)
    meas_pos_elbow    = np.zeros(n)
    meas_vel_elbow    = np.zeros(n)
    meas_tau_elbow    = np.zeros(n)
    des_tau_elbow     = np.zeros(n)

    tau = [0.0, 0.0]

    i = 0
    meas_dt = 0.0
    meas_time = 0.0
    max_exec_freq = 0.0
    min_exec_freq = math.inf
    avg_exec_freq = 0.0
    print("Free-Fall Loop Started!")

    while meas_time < tf and i < n:
        start_loop = time.time()

        measured_position = c.get_position(session_token)
        measured_velocity = c.get_velocity(session_token)
        measured_torque   = c.get_torque(session_token)

        x_now = np.concatenate([measured_position, measured_velocity])

        if i == 0:
            print("Initial state x:", x_now)

        # Always command zero torque.
        c.set_torque(tau, session_token)

        meas_time_vec[i]     = meas_time
        meas_pos_shoulder[i] = measured_position[0]
        meas_vel_shoulder[i] = measured_velocity[0]
        meas_tau_shoulder[i] = measured_torque[0]
        des_tau_shoulder[i]  = 0.0
        meas_pos_elbow[i]    = measured_position[1]
        meas_vel_elbow[i]    = measured_velocity[1]
        meas_tau_elbow[i]    = measured_torque[1]
        des_tau_elbow[i]     = 0.0

        i += 1
        exec_time = time.time() - start_loop
        min_exec_freq = min(min_exec_freq, 1.0 / exec_time)
        max_exec_freq = max(max_exec_freq, 1.0 / exec_time)
        avg_exec_freq = avg_exec_freq + 1.0 / exec_time

        while time.time() - start_loop < dt:
            pass
        meas_dt = time.time() - start_loop
        meas_time += meas_dt

    print("Free-Fall Loop Ended!")
    avg_exec_freq = avg_exec_freq / float(max(i, 1))
    print(
        "Finished",
        "- avg exec frequency:", avg_exec_freq,
        " - min exec frequency:", min_exec_freq,
        " - max exec frequency:", max_exec_freq,
    )

    download_url = c.stop_experiment(session_token)

    if record:
        filename = wget.download(download_url, ".")
        vod_filepath = f"{Path(filename).stem}.mp4"
        convert_flv_to_mp4(f"{filename}", vod_filepath)

    # Trim to the samples actually collected.
    meas_time_vec     = meas_time_vec[:i]
    meas_pos_shoulder = meas_pos_shoulder[:i]
    meas_pos_elbow    = meas_pos_elbow[:i]
    meas_vel_shoulder = meas_vel_shoulder[:i]
    meas_vel_elbow    = meas_vel_elbow[:i]
    meas_tau_shoulder = meas_tau_shoulder[:i]
    meas_tau_elbow    = meas_tau_elbow[:i]
    des_tau_shoulder  = des_tau_shoulder[:i]
    des_tau_elbow     = des_tau_elbow[:i]

    t_values = meas_time_vec
    x_values = np.vstack((
        meas_pos_shoulder,
        meas_pos_elbow,
        meas_vel_shoulder,
        meas_vel_elbow,
    )).T
    tau_values = np.vstack((
        meas_tau_shoulder,
        meas_tau_elbow,
    )).T
    des_tau_values = np.vstack((
        des_tau_shoulder,
        des_tau_elbow,
    )).T

    print("Returned valid samples:", i)
    if i > 0:
        print("Final measured time:", t_values[-1])

    return t_values, x_values, tau_values, des_tau_values, vod_filepath, cell_id

In [11]:
dt = 0.005
tf = 10

x0 = [2*np.pi/3, -np.pi/2,0,0]
T_exp, X_exp, U_exp, U_des_exp, vod_path, cell_id = run_free_fall(tf=tf, dt=dt, x0 = x0,
                  user_token=user_token,
                  experiment_type="DoublePendulum",
                  preparation_time=4.0,
                  record=False)

Successfully reserved cell on the server
Livestream url:  http://cloudpendulum.m2.chalmers.se:5080/live/viewer.jsp?host=cloudpendulum.m2.chalmers.se&stream=camera_171
Starting experiment in 3 seconds
Starting experiment in 2 seconds
Starting experiment in 1 seconds
Starting experiment
Your session token is: 2075658230779944448
Detected cell/camera: 171
n:2000
Free-Fall Loop Started!
Initial state x: [ 1.72846079 -1.57271385  0.00376746  0.22686647]
Free-Fall Loop Ended!
Finished - avg exec frequency: 812.9085633183969  - min exec frequency: 21.893453319274656  - max exec frequency: 1384.2587458745875
Returned valid samples: 1933
Final measured time: 9.99792742729187


In [12]:
# Get data files
measured_csv_data = np.array([T_exp, np.asarray(X_exp).T[0], np.asarray(X_exp).T[1], np.asarray(X_exp).T[2], np.asarray(X_exp).T[3], np.asarray(U_exp).T[0],np.asarray(U_exp).T[1]]).T
np.savetxt(f"data/{cell_id}_data_103.csv", measured_csv_data, delimiter = ',', header="time,pos_meas1,pos_meas2,vel_meas1,vel_meas2,tau_meas1,tau_meas2", comments="")